# EX-0625 — Démonstration G15 : un seul bouton
Cliphton Marseille et Igor Gaitan · Cerise sur le ghetto

Cliquer sur ▶ à gauche du seul bloc ci-dessous. Si demandé, saisir la clé OpenRouter du cours dans la zone masquée. La clé ne doit jamais être collée dans le code.

Premier essai : laisser `CHOIX_TEST = 2`. Deuxième essai : mettre `CHOIX_TEST = 3` en haut du bloc, puis cliquer à nouveau sur ▶.

Le code installe les dépendances manquantes, prépare la fonction et appelle l’IA. Il compare sa réponse à une référence définie avant l’appel. Modèle : qwen/qwen3.8-flash sur OpenRouter par défaut.

Assistance ChatGPT/Codex déclarée. Source : CoursIA, 03_Structured_Outputs.ipynb, EX-0625.


In [1]:
# UN SEUL BOUTON ▶ POUR TOUT LANCER
# 2 = Lucas sans email ; 3 = Lucas avec email
CHOIX_TEST = 2
import importlib.util, subprocess, sys
manquants = [paquet for module, paquet in [("openai", "openai"), ("dotenv", "python-dotenv")] if importlib.util.find_spec(module) is None]
if manquants:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *manquants])
import pydantic
if int(pydantic.__version__.split(".")[0]) < 2:
    raise RuntimeError("Pydantic 2 requis : installer pydantic>=2 et redémarrer la session.")
import os, json
from pathlib import Path
from typing import List, Optional
from pydantic import BaseModel, Field, ValidationError, conint
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(dotenv_path=Path.cwd() / ".env")
cle = os.getenv("OPENROUTER_API_KEY") or os.getenv("OPENAI_API_KEY")
if not cle:
    from getpass import getpass
    cle = getpass("Clé API du cours (saisie masquée) : ")
    if not cle.strip():
        raise RuntimeError("La clé API du cours est nécessaire pour appeler le modèle.")
client = OpenAI(api_key=cle, base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"), max_retries=0, timeout=60)
DEFAULT_MODEL = os.getenv("OPENAI_MODEL", "qwen/qwen3.8-flash")
def add_additional_properties_false(schema: dict) -> dict:
    """
    Ajoute récursivement additionalProperties: False à tous les objets du schema
    ET s'assure que tous les champs sont dans required (requis par OpenAI).
    
    OpenAI Structured Outputs exige:
    1. additionalProperties: False sur tous les objets
    2. Tous les champs dans 'required' (même ceux avec defaults)
    """
    if not isinstance(schema, dict):
        return schema
    
    # Traiter les définitions $defs d'abord (Pydantic 2.x les génère)
    if "$defs" in schema:
        for def_name, def_schema in schema["$defs"].items():
            add_additional_properties_false(def_schema)
    
    # Si c'est un objet, ajouter additionalProperties: False
    # ET s'assurer que tous les champs sont dans required
    if schema.get("type") == "object":
        schema["additionalProperties"] = False
        # Ajouter tous les champs de properties dans required
        if "properties" in schema:
            all_props = list(schema["properties"].keys())
            schema["required"] = all_props
            # Traiter récursivement les propriétés
            for prop_name, prop_schema in schema["properties"].items():
                add_additional_properties_false(prop_schema)
    
    # Traiter les items d'arrays
    if schema.get("type") == "array" and "items" in schema:
        add_additional_properties_false(schema["items"])
    
    # Traiter anyOf, allOf, oneOf
    for key in ["anyOf", "allOf", "oneOf"]:
        if key in schema:
            for item in schema[key]:
                add_additional_properties_false(item)
    
    return schema

def extract_structured(prompt: str, model_class: type[BaseModel], model: str = None):
    """
    Fonction utilitaire pour extraction structurée avec gestion d'erreurs.

    Args:
        prompt: Prompt utilisateur
        model_class: Classe Pydantic pour validation
        model: Modèle OpenAI à utiliser (défaut: DEFAULT_MODEL)

    Returns:
        Instance validée de model_class ou None en cas d'erreur
    """
    if model is None:
        model = DEFAULT_MODEL
    
    try:
        # Générer le schema et ajouter additionalProperties: False (requis par OpenAI)
        schema = model_class.model_json_schema()
        schema = add_additional_properties_false(schema)

        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": model_class.__name__.lower(),
                    "strict": True,
                    "schema": schema
                }
            }
        )
        return model_class.model_validate_json(response.choices[0].message.content)
    except ValidationError as e:
        print(f"Erreur de validation Pydantic: {e}")
        return None
    except Exception as e:
        print(f"Erreur API: {e}")
        return None


from typing import Optional, Literal, Union
from collections import Counter
from pydantic import BaseModel, Field
import json
import unicodedata

class ContactExtrait(BaseModel):
    nom: str
    telephone: str
    email: Optional[str] = Field(default=None, description="null si l'information n'apparait pas dans le texte")
    entreprise: Optional[str] = Field(default=None, description="null si l'information n'apparait pas dans le texte")
    ville: Optional[str] = Field(default=None, description="null si l'information n'apparait pas dans le texte")

texte_complet = ("Bonjour, je m'appelle Ines Benali, joignable au 06 98 76 54 32. "
                 "Email : ines.benali@exemple.fr, societe Atelier Numerique, Lyon. Je souhaite un devis.")
texte_lacunaire = "Bonjour, je m'appelle Ines Benali, joignable au 06 98 76 54 32. Je souhaite un devis."
texte_lucas = "Bonjour, je suis Lucas Martin de l'entreprise Nova, à Bordeaux. Vous pouvez me rappeler au 06 12 34 56 78. Je souhaite un devis."
consigne = "Extrais uniquement les informations explicites du texte, sans inference. Utilise null pour les informations absentes. Texte : "

# La référence est définie AVANT l'appel à l'IA, à partir de chaque texte.
# Pour ajouter un autre message, indiquer aussi ses valeurs attendues.
cas_tests = [
    ("Ines : message complet", texte_complet,
     dict(nom="Ines Benali", telephone="06 98 76 54 32", email="ines.benali@exemple.fr", entreprise="Atelier Numerique", ville="Lyon")),
    ("Ines : trois informations absentes", texte_lacunaire,
     dict(nom="Ines Benali", telephone="06 98 76 54 32", email=None, entreprise=None, ville=None)),
    ("Lucas : email absent", texte_lucas,
     dict(nom="Lucas Martin", telephone="06 12 34 56 78", email=None, entreprise="Nova", ville="Bordeaux")),
    ("Lucas : email ajouté", texte_lucas + " Mon email est lucas.martin@exemple.fr.",
     dict(nom="Lucas Martin", telephone="06 12 34 56 78", email="lucas.martin@exemple.fr", entreprise="Nova", ville="Bordeaux")),
]

def normaliser(valeur):
    if valeur is None:
        return None
    texte = unicodedata.normalize("NFKD", valeur.casefold())
    return "".join(c for c in texte if c.isalnum())

def verifier_contact(resultat, reference):
    if resultat is None:
        return {"extraction_reussie": False}
    valeurs = resultat.model_dump()
    absents_attendus = [champ for champ, valeur in reference.items() if valeur is None]
    return {
        "extraction_reussie": True,
        "valeurs": valeurs,
        "champs_renseignes": sum(v is not None for v in valeurs.values()),
        "absents_attendus": absents_attendus,
        "absents_corrects": sum(valeurs[c] is None for c in absents_attendus),
        "champs_conformes": {c: normaliser(valeurs[c]) == normaliser(v) for c, v in reference.items()},
    }

# Un seul cas par lancement pour la démonstration, sans relancer tout le cours.
# Choisir 0 ou 1 pour l'exercice initial, puis 2 et 3 pour Lucas.
etiquette, texte, reference = cas_tests[CHOIX_TEST]
print("TEST :", etiquette)
print("MESSAGE :", texte)
resultat = extract_structured(consigne + texte, ContactExtrait)
bilan = verifier_contact(resultat, reference)
if not bilan["extraction_reussie"]:
    print("ECHEC D'EXTRACTION : vérifier la configuration ou l'erreur affichée plus haut.")
else:
    print("REPONSE DE L'IA :")
    print(json.dumps(bilan["valeurs"], ensure_ascii=False, indent=2))
    print("Champs renseignés :", bilan["champs_renseignes"])
    total_absents = len(bilan["absents_attendus"])
    if total_absents:
        print("Champs attendus absents :", ", ".join(bilan["absents_attendus"]))
        print(f"Absences respectées : {bilan['absents_corrects']}/{total_absents}")
    else:
        print("Aucune information attendue absente pour ce test.")
    print("VERIFICATION DES CINQ CHAMPS :")
    for champ, conforme in bilan["champs_conformes"].items():
        print(champ, ":", "OK" if conforme else "A VERIFIER", "/ attendu :", repr(reference[champ]))
    print("Tous les champs conformes :", all(bilan["champs_conformes"].values()))
# Comparaison à une référence explicite, avec normalisation simple de l'écriture.
# Elle ne remplace pas une vérification sémantique générale.


TEST : Lucas : email absent
MESSAGE : Bonjour, je suis Lucas Martin de l'entreprise Nova, à Bordeaux. Vous pouvez me rappeler au 06 12 34 56 78. Je souhaite un devis.
REPONSE DE L'IA :
{
  "nom": "Lucas Martin",
  "telephone": "06 12 34 56 78",
  "email": null,
  "entreprise": "Nova",
  "ville": "Bordeaux"
}
Champs renseignés : 4
Champs attendus absents : email
Absences respectées : 1/1
VERIFICATION DES CINQ CHAMPS :
nom : OK / attendu : 'Lucas Martin'
telephone : OK / attendu : '06 12 34 56 78'
email : OK / attendu : None
entreprise : OK / attendu : 'Nova'
ville : OK / attendu : 'Bordeaux'
Tous les champs conformes : True


## Résultat réel enregistré du deuxième essai
Exécution du 16 octobre 2026 avec le même bloc et `CHOIX_TEST = 3`. Les deux essais ont été exécutés en Python avec la configuration du notebook et l’API du cours, modèle qwen/qwen3.8-flash. Ces résultats sont enregistrés pour consultation sur GitHub. Deux appels IA ont été effectués.

```text
TEST : Lucas : email ajouté
MESSAGE : Bonjour, je suis Lucas Martin de l'entreprise Nova, à Bordeaux. Vous pouvez me rappeler au 06 12 34 56 78. Je souhaite un devis. Mon email est lucas.martin@exemple.fr.
REPONSE DE L'IA :
{
  "nom": "Lucas Martin",
  "telephone": "06 12 34 56 78",
  "email": "lucas.martin@exemple.fr",
  "entreprise": "Nova",
  "ville": "Bordeaux"
}
Champs renseignés : 5
Aucune information attendue absente pour ce test.
VERIFICATION DES CINQ CHAMPS :
nom : OK / attendu : 'Lucas Martin'
telephone : OK / attendu : '06 12 34 56 78'
email : OK / attendu : 'lucas.martin@exemple.fr'
entreprise : OK / attendu : 'Nova'
ville : OK / attendu : 'Bordeaux'
Tous les champs conformes : True
```

Pour refaire le test en direct, choisir 2 ou 3 en haut du bloc et cliquer sur ▶ dans Colab. Les réponses futures peuvent varier.
